# Prepare disjoint splits and tokenize

See README.md for prerequisites. Restart the kernel and run this notebook from the repository root.


In [ ]:
import json
import os
import shutil
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [ ]:
# ── Configuration ──
MODEL_NAME = "answerdotai/ModernBERT-base"
DATASET_FOLDER = "data/model_ready"
OUTPUT_FOLDER = "data/tokenized"
JSON_PATH = "reports/recommended_tokenization.json"

TEST_SIZE = 0.15
VALID_SIZE = 0.15
RANDOM_STATE = 42

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Configuration set")
print(f"  Model        : {MODEL_NAME}")
print(f"  Input folder : {DATASET_FOLDER}")
print(f"  Output folder: {OUTPUT_FOLDER}")
print(f"  Split        : {1-TEST_SIZE-VALID_SIZE:.0%} train / {VALID_SIZE:.0%} val / {TEST_SIZE:.0%} test")

In [ ]:
# Predeclared limits, not selected by looking at held-out test examples.
# To change these, use training/validation data only and record a new experiment.
recommended_lengths = {"sentiment":128, "intent":128, "root_cause":256, "risk":256}
print(recommended_lengths)


In [ ]:
# Start from task labels; fit a compact mapping AFTER exclusions.
from pathlib import Path
from training_quality import clean_task, split_task
import hashlib
provenance_path = Path(DATASET_FOLDER) / "dataset_manifest.json"
if not provenance_path.is_file():
    raise FileNotFoundError("Run updated notebook 5 after auditing v3 labels; dataset_manifest.json is missing")
provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
if provenance["taxonomy_sha256"] != hashlib.sha256(Path("taxonomy.json").read_bytes()).hexdigest():
    raise ValueError("Taxonomy changed since dataset preparation; rebuild reviewed data first")
for filename, expected in provenance["files"].items():
    if hashlib.sha256((Path(DATASET_FOLDER) / filename).read_bytes()).hexdigest() != expected:
        raise ValueError(f"{filename} changed since notebook 5; rebuild datasets first")
frames = {}
label_names_by_task = {}
quality_dir = Path("reports/training_quality")
quality_dir.mkdir(parents=True, exist_ok=True)
MIN_CLASS_COUNT = 10
for task_name in ("sentiment", "intent", "root_cause", "risk"):
    frame = pd.read_csv(Path(DATASET_FOLDER) / f"{task_name}_dataset.csv")
    target = "risk_band" if task_name == "risk" else task_name
    excluded_labels = ("noise",) if task_name == "intent" else (("other", "unreadable_text") if task_name == "root_cause" else ())
    frame, excluded = clean_task(frame, target, min_count=MIN_CLASS_COUNT, excluded=excluded_labels)
    names = sorted(frame[target].unique().tolist())
    if task_name == "risk":
        order = ["NEGLIGIBLE", "LOW", "MODERATE", "HIGH", "CRITICAL"]
        if not set(names).issubset(order): raise ValueError("Unknown risk band")
        names = [name for name in order if name in names]
    label_names_by_task[task_name] = names
    frame[f"{task_name}_label"] = frame[target].map({name:i for i,name in enumerate(names)})
    frames[task_name] = frame
    excluded.to_csv(quality_dir / f"{task_name}_tokenization_excluded.csv", index=False)
    print(task_name, "retained classes:", len(names), "excluded rows:", len(excluded))
intent_df, sentiment_df, root_df, risk_df = (frames[name] for name in ("intent", "sentiment", "root_cause", "risk"))


In [ ]:
# Every output class has training examples; no stale encoder can leave zero-count heads.
for task_name, frame in frames.items():
    ids = sorted(frame[f"{task_name}_label"].unique().tolist())
    assert ids == list(range(len(label_names_by_task[task_name])))
    assert frame[f"{task_name}_label"].value_counts().min() >= MIN_CLASS_COUNT


In [ ]:
# Create deterministic, disjoint training, validation and test splits.
splits = {}
for task_name, frame in frames.items():
    target = "risk_band" if task_name == "risk" else task_name
    splits[task_name] = split_task(frame, target, seed=RANDOM_STATE, test_size=TEST_SIZE, valid_size=VALID_SIZE)
    print(task_name, {name:len(part) for name,part in splits[task_name].items()})


In [ ]:
# ── Load tokenizer once ──
print(f"Loading tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer ready (vocab size: {tokenizer.vocab_size:,}, model max: {tokenizer.model_max_length})")

# Verify recommended lengths don't exceed model limit
for task, length in recommended_lengths.items():
    if length > tokenizer.model_max_length:
        raise ValueError(f"{task}: recommended max_length ({length}) exceeds model limit ({tokenizer.model_max_length})")
    print(f"  OK  {task:<12s}  max_length={length}")

In [ ]:
# ── Tokenize all tasks and save to disk ──

TASK_CONFIGS = {
    "intent":     {"text_col": "text", "label_col": "intent_label"},
    "sentiment":  {"text_col": "text", "label_col": "sentiment_label"},
    "root_cause": {"text_col": "text", "label_col": "root_cause_label"},
    "risk":       {"text_col": "text", "label_col": "risk_label"},
}

for task_name, config in TASK_CONFIGS.items():
    max_len = recommended_lengths[task_name]
    text_col = config["text_col"]
    label_col = config["label_col"]
    
    print(f"\nTokenizing {task_name.upper()} (max_length={max_len}) ...")
    
    split_datasets = {}
    
    for split_name in ["train", "val", "test"]:
        split_df = splits[task_name][split_name].reset_index(drop=True)
        texts = split_df[text_col].astype(str).tolist()
        labels = split_df[label_col].tolist()
        
        # Tokenize in batches of 512
        all_input_ids = []
        all_attention_mask = []
        
        for start in range(0, len(texts), 512):
            batch = texts[start : start + 512]
            enc = tokenizer(
                batch,
                max_length=max_len,
                truncation=True,
                padding=False,        # DataCollatorWithPadding handles this during training
                return_tensors=None,
            )
            all_input_ids.extend(enc["input_ids"])
            all_attention_mask.extend(enc["attention_mask"])
        
        ds = Dataset.from_dict({
            "input_ids": all_input_ids,
            "attention_mask": all_attention_mask,
            "labels": labels,  # must be 'labels' (with s) for HuggingFace Trainer
        })
        
        hf_split = "validation" if split_name == "val" else split_name
        split_datasets[hf_split] = ds
    
    dd = DatasetDict(split_datasets)
    
    # Save to disk (clear old data first)
    save_path = os.path.join(OUTPUT_FOLDER, task_name)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    dd.save_to_disk(save_path)
    tokenizer.save_pretrained(os.path.join(save_path, "tokenizer"))
    manifest = {
        "schema_version": 2, "task": task_name, "base_model": MODEL_NAME,
        "label_names": label_names_by_task[task_name], "max_length": int(max_len),
        "text_preprocessing": "as_is", "random_seed": RANDOM_STATE,
        "min_class_count": MIN_CLASS_COUNT,
        "taxonomy_sha256": provenance["taxonomy_sha256"],
        "split_text_sha256": {name: __import__("hashlib").sha256("\n".join(part["_text_key"]).encode()).hexdigest() for name,part in splits[task_name].items()},
        "source_sha256": __import__("hashlib").sha256((Path(DATASET_FOLDER) / f"{task_name}_dataset.csv").read_bytes()).hexdigest(),
        "split_sizes": {name: len(dataset) for name, dataset in dd.items()},
    }
    with open(os.path.join(save_path, "training_manifest.json"), "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2)

    
    print(f"  Saved {task_name} -> {save_path}")
    print(f"    train={len(dd['train']):,}  val={len(dd['validation']):,}  test={len(dd['test']):,}")

print("\nAll four datasets tokenized and saved.")
print("\nTo load in training notebook:")
print("  from datasets import DatasetDict")
print(f"  dd = DatasetDict.load_from_disk('{OUTPUT_FOLDER}/intent')")
print("  # dd['train'], dd['validation'], dd['test']")